In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import io
import base64
import json
import gc
import pymongo

2024-11-19 14:30:20.781775: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-19 14:30:20.804767: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-19 14:30:21.119017: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
def process_image(base64_str, target_size=(64, 64)):
    try:
        image_data = base64.b64decode(base64_str)
        image = Image.open(io.BytesIO(image_data)).convert('RGB')
        image = image.resize(target_size)
        return np.array(image) / 255.0
    except Exception as e:
        print(f"Fehler bei der Bildverarbeitung: {e}")
        return None

In [3]:
def load_data(limit_per_user=10000):
    print("Lade Daten aus der Datenbank...")
    mongo_uri = os.getenv('MONGO_URI', 'mongodb://localhost:27017/fingerprintDB')
    client = pymongo.MongoClient(mongo_uri)
    db = client.get_default_database()

    canvassamples_collection = db['canvassamples']
    canvassamples_cursor = canvassamples_collection.find()
    canvassamples_data = list(canvassamples_cursor)
    canvassamples_df = pd.DataFrame(canvassamples_data)

    fingerprints_collection = db['fingerprints']
    fingerprints_cursor = fingerprints_collection.find()
    fingerprints_data = list(fingerprints_cursor)
    fingerprints_df = pd.DataFrame(fingerprints_data)

    merged_df = pd.merge(canvassamples_df, fingerprints_df, left_on='fingerprintId', right_on='_id', suffixes=('_sample', '_fingerprint'))

    user_ids = merged_df['username'].unique()

    user_dfs = {}
    for user_id in user_ids:
        user_df = merged_df[merged_df['username'] == user_id]
        if len(user_df) > limit_per_user:
            user_df = user_df.sample(n=limit_per_user, random_state=42)
        user_dfs[user_id] = user_df

    print("Daten erfolgreich geladen.")
    return user_dfs

user_dfs = load_data(limit_per_user=10000)

Lade Daten aus der Datenbank...
Daten erfolgreich geladen.


In [4]:
def process_user_data(user_dfs, batch_size=1000):
    user_data = {}
    for user_id, df in user_dfs.items():
        try:
            X = []
            y = []
            for start in range(0, len(df), batch_size):
                end = start + batch_size
                batch = df.iloc[start:end]
                X_batch = [process_image(x) for x in batch['sampleData']]
                y_batch = (batch['username'] == 'user_1').astype(int).values
                X.extend(X_batch)
                y.extend(y_batch)
                gc.collect()  # Speicher freigeben
            user_data[user_id] = (np.array(X), np.array(y))
        except Exception as e:
            print(f"Fehler bei der Datenverarbeitung für Benutzer {user_id}: {e}")
    return user_data

user_data = process_user_data(user_dfs)

In [ ]:
import os
from tensorflow.keras.models import load_model

def load_all_cnn_models(base_dir="Modelle"):
    print("Lade alle CNN-Modelle...")
    models = {}
    for model_dir in os.listdir(base_dir):
        dir_path = os.path.join(base_dir, model_dir)
        if os.path.isdir(dir_path) and model_dir.startswith("CNN"):
            for model_file in os.listdir(dir_path):
                if model_file.endswith('.h5'):
                    model_path = os.path.join(dir_path, model_file)
                    try:
                        model = load_model(model_path)
                        models[f"{model_dir}/{model_file}"] = model
                        print(f"Modell geladen: {model_dir}/{model_file}")
                    except Exception as e:
                        print(f"Fehler beim Laden von {model_file}: {e}")
    if not models:
        raise ValueError("Keine CNN-Modelle gefunden. Überprüfen Sie das Verzeichnis und die Dateinamen.")
    print("Alle Modelle erfolgreich geladen.")
    return models

# Modelle laden
print("Lade Modelle...")
cnn_models = load_all_cnn_models()

Lade Modelle...
Lade alle CNN-Modelle...


2024-11-19 14:32:27.301722: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-11-19 14:32:27.301833: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-11-19 14:32:27.315455: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

Modell geladen: CNN Modell/CNN Modell_cnn_model.h5
Modell geladen: CNN RGB/CNN RGB_cnn_model.h5
Modell geladen: CNN_Negative/model_experiment_2_ResNet.h5
Modell geladen: CNN_Negative/model_experiment_3_MobileNetV2.h5
Modell geladen: CNN_Negative/model_experiment_7_CNN.h5
Modell geladen: CNN_Negative/model_experiment_1_CNN.h5
Modell geladen: CNN_Negative/model_experiment_4_CNN.h5
Modell geladen: CNN_Negative/model_experiment_6_MobileNetV2.h5
Modell geladen: CNN_Negative/model_experiment_8_ResNet.h5
Modell geladen: CNN_Negative/model_experiment_10_CNN.h5
Modell geladen: CNN_Negative/model_experiment_9_MobileNetV2.h5


In [ ]:
def evaluate_model(model, X_test, y_test):
    print("Evaluierung des Modells...")
    try:
        y_pred_prob = model.predict(X_test).flatten()
        y_pred = (y_pred_prob > 0.5).astype(int)

        conf_matrix = confusion_matrix(y_test, y_pred)
        class_report = classification_report(y_test, y_pred, target_names=["Andere Nutzer", "Beispielnutzer"], labels=[0, 1], zero_division=0)
        fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
        roc_auc = auc(fpr, tpr)

        return conf_matrix, class_report, fpr, tpr, roc_auc, y_test, y_pred, y_pred_prob
    except Exception as e:
        print(f"Fehler bei der Evaluierung des Modells: {e}")
        raise

def evaluate_all_models(models, user_data, example_user_id):
    results = {}
    for model_name, model in models.items():
        print(f"Evaluierung des Modells: {model_name}")
        for user_id, (X_test, y_test) in user_data.items():
            if user_id == example_user_id:
                continue
            print(f"Evaluierung gegen Benutzer {user_id}...")
            try:
                conf_matrix, class_report, fpr, tpr, roc_auc, y_test, y_pred, y_pred_prob = evaluate_model(model, X_test, y_test)
                results[f"{model_name}_{user_id}"] = {
                    'conf_matrix': conf_matrix.tolist(),
                    'class_report': class_report,
                    'fpr': fpr.tolist(),
                    'tpr': tpr.tolist(),
                    'roc_auc': roc_auc,
                    'y_test': y_test.tolist(),
                    'y_pred': y_pred.tolist(),
                    'y_pred_prob': y_pred_prob.tolist()
                }
                print(f"Erfolgreiche Evaluierung für Modell {model_name} gegen Benutzer {user_id}")
            except Exception as e:
                print(f"Fehler bei der Evaluierung von Modell {model_name} gegen Benutzer {user_id}: {e}")
    if not results:
        raise ValueError("Keine Ergebnisse vorhanden. Überprüfen Sie die Eingabedaten und Modelle.")
    return results

example_user_id = 'benutzername_1'
evaluation_results = evaluate_all_models(cnn_models, user_data, example_user_id)

# Ergebnisse speichern
with open('cnn_evaluation_results.json', 'w') as f:
    json.dump(evaluation_results, f)

print("Ergebnisse wurden in 'cnn_evaluation_results.json' gespeichert.")

# Ergebnisse visualisieren
for model_user, result in evaluation_results.items():
    conf_matrix = np.array(result['conf_matrix'])
    class_report = result['class_report']
    fpr = np.array(result['fpr'])
    tpr = np.array(result['tpr'])
    roc_auc = result['roc_auc']
    y_test = np.array(result['y_test'])
    y_pred = np.array(result['y_pred'])
    y_pred_prob = np.array(result['y_pred_prob'])

    # Konfusionsmatrix visualisieren
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Andere Nutzer", "Beispielnutzer"],
                yticklabels=["Andere Nutzer", "Beispielnutzer"])
    plt.title(f'Konfusionsmatrix - {model_user}')
    plt.xlabel('Vorhergesagte Klasse')
    plt.ylabel('Tatsächliche Klasse')
    plt.show()

    # ROC-Kurve darstellen
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic (ROC) - {model_user}')
    plt.legend(loc="lower right")
    plt.show()

    # Vorhersagewahrscheinlichkeiten visualisieren
    results_df = pd.DataFrame({
        'y_test': y_test,
        'y_pred_prob': y_pred_prob,
        'y_pred': y_pred
    })

    results_df['Fehler'] = np.where(
        (results_df['y_pred'] == 1) & (results_df['y_test'] == 0), 'False Positive',
        np.where(
            (results_df['y_pred'] == 0) & (results_df['y_test'] == 1), 'False Negative',
            'Korrekt'
        )
    )

    selected_users = results_df.index.unique()[:4]

    plt.figure(figsize=(15, 8))

    for user in selected_users:
        user_data = results_df.loc[user]
        plt.plot(
            np.arange(len(user_data)),
            user_data['y_pred_prob'],
            label=f'Nutzer: {user}'
        )

    plt.xlabel('Samples', fontsize=14)
    plt.ylabel('Vorhersagewahrscheinlichkeit', fontsize=14)
    plt.title(f'Modellkonfidenz pro Nutzer - {model_user}', fontsize=16)
    plt.legend()
    plt.show()

In [ ]:
# Option 1: Speicher der spezifischen GPU bereinigen
import torch
torch.cuda.empty_cache()  # Cache leeren
torch.cuda.memory.empty_cache()  # Gründlichere Bereinigung

# Option 2: Speicher aller GPUs bereinigen und CUDA zurücksetzen
def clear_gpu_memory():
    import torch
    import gc
    
    # PyTorch-Cache leeren
    torch.cuda.empty_cache()
    
    # Alle PyTorch-Tensoren auf die CPU verschieben und löschen
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj):
                obj = obj.cpu()
        except: 
            pass
    
    # Garbage Collector ausführen
    gc.collect()
    
    # CUDA zurücksetzen
    torch.cuda.empty_cache()
    torch.cuda.memory.empty_cache()